# NB4 · Fixed-hip balance with finite traction

Use the four-state v2 example as a regression point and design a separate
three-state regulator for balance without an anchor. Compare both on a sampled
nonlinear wheel-moment pendulum with limited effort and finite tangential force.
The simulator terminates when fixed axle height becomes invalid. The regularized
friction law and its explicit settings make this a reproducible comparison, with
the v2 nonlinear table as historical reference rather than a test oracle.
[DUK v3 chapter 06](https://app.notion.com/p/4131f3cc033281c4a7afcb8b23228bba).

In [ ]:
import hashlib
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from duk_core.modeling import load_leg_geometry_json

repo = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "robot/config/physical/duk_leg_geometry.json").is_file()
)
geometry_path = repo / "robot/config/physical/duk_leg_geometry.json"
fixture_path = repo / "research/experiments/duk-v3/synthetic.json"
geometry = load_leg_geometry_json(geometry_path)
fixture = json.loads(fixture_path.read_text())
assert fixture["schema_version"] == 1 and fixture["status"] == "synthetic_research_only"
print(
    "Working tree dirty:",
    bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=repo)),
)
print(
    "Commit",
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip(),
)
for path in (geometry_path, fixture_path):
    print(path.relative_to(repo), hashlib.sha256(path.read_bytes()).hexdigest())
print(fixture["source"])

In [ ]:
from duk_core.control.lqr import bryson_weights, discrete_lqr, linearize
from duk_core.control.sagittal_balance import balance_command
from duk_core.modeling import build_planar_biped_model
from duk_core.modeling.planar_numeric import NumericPlanarBiped
from duk_core.modeling.rolling_reduction import PrescribedSagittal
from duk_core.modeling.wheeled_pendulum import WheeledPendulum
from duk_core.simulation.balance import simulate_balance

p = fixture["v2_balance"]
s = fixture["balance_experiment"]
keys = (
    "wheel_mass_kg",
    "wheel_inertia_kg_m2",
    "upper_mass_kg",
    "upper_inertia_kg_m2",
    "length_m",
    "radius_m",
    "gravity_m_s2",
)
example = WheeledPendulum(**{key: p[key] for key in keys})
period = p["sample_period_s"]
q4, r = bryson_weights(p["state_scales"], p["torque_scale_N_m"])
a4, b4 = example.linearization()
position = discrete_lqr(a4, b4, q4, r, period)
balance = discrete_lqr(a4[1:, 1:], b4[1:], q4[1:, 1:], r, period)
np.testing.assert_allclose(
    position.gain[0], [-5.640, -7.383, -30.327, -4.254], atol=0.001, rtol=0
)
for fit in (position, balance):
    assert fit.riccati_residual < 1e-10 and fit.spectral_radius < 1
assert balance_command(balance.gain, [0.1, 0, 0, 0]) == 0.0
assert abs(balance_command(position.gain, [0.1, 0, 0, 0], anchor=0.0)) > 0.5
print("Position hold:", position.gain, position.spectral_radius)
print("Balance only:", balance.gain, balance.spectral_radius)

planar = NumericPlanarBiped(build_planar_biped_model(geometry), fixture["planar"])
q0 = np.deg2rad(95)
offset = planar.equivalent_upper_body(q0).angle_from_vertical_rad
reference = PrescribedSagittal(planar, offset)
params = WheeledPendulum.from_biped(planar, q0)
a_expected, b_expected = params.linearization()
a_numeric, b_numeric = linearize(
    lambda x, torque: reference.evaluate(x, torque, (q0, 0.0, 0.0)).derivative,
    np.zeros(4),
    0.0,
)
np.testing.assert_allclose(a_numeric, a_expected, rtol=1e-6, atol=1e-7)
np.testing.assert_allclose(b_numeric, b_expected, rtol=1e-6, atol=1e-7)
print(
    "Independent 7 DOF linearization errors:",
    np.max(abs(a_expected - a_numeric)),
    np.max(abs(b_expected - b_numeric)),
)

In [ ]:
def run(
    friction,
    source,
    budget=None,
    *,
    angle=s["initial_lean_deg"],
    delay=0,
    noise=False,
    hold=True,
    plant=example,
    motor_bound=None,
):
    return simulate_balance(
        plant,
        position.gain if hold else balance.gain,
        np.deg2rad(angle),
        period=period,
        duration=s["duration_s"],
        friction=friction,
        motor_bound=p["total_torque_limit_N_m"] if motor_bound is None else motor_bound,
        slip_scale=s["slip_scale_m_s"],
        fall_angle=np.deg2rad(s["fall_angle_deg"]),
        feedback=source,
        traction_budget=budget,
        anchor=0.0 if hold else None,
        delay_samples=delay,
        noise_seed=s["sensor_seed"],
        accel_noise=s["acceleration_noise_m_s2"],
        pitch_noise=s["pitch_noise_rad"] if noise else 0.0,
        slip_threshold=s["slip_threshold_m_s"],
    )


cases = {
    "dry, wheel odometry": run(0.8, "odometry"),
    "low grip, wheel odometry": run(0.3, "odometry"),
    "low grip, simulator truth": run(0.3, "truth"),
    "low grip, wheel odometry + traction": run(0.3, "odometry", 0.25),
    "low grip, gated + traction + delay": run(0.3, "gated", 0.25, delay=1, noise=True),
}
for name, trace in cases.items():
    assert np.max(abs(trace.applied)) <= p["total_torque_limit_N_m"] + 1e-12
    print(name, trace.outcome, "peak slip (m/s)", round(np.max(abs(trace.slip)), 3))
assert abs(cases["dry, wheel odometry"].requested[0]) > p["total_torque_limit_N_m"]
assert cases["dry, wheel odometry"].outcome == "completed"
assert np.max(abs(cases["low grip, wheel odometry + traction"].slip)) < np.max(
    abs(cases["low grip, wheel odometry"].slip)
)

fig, axes = plt.subplots(2, 2, figsize=(11, 6), constrained_layout=True)
for name, trace in cases.items():
    axes[0, 0].plot(trace.time, np.rad2deg(trace.state[:, 2]), label=name)
    axes[0, 1].plot(trace.time, trace.slip)
    utilization = abs(trace.tangent) / np.maximum(trace.normal, 1e-12)
    axes[1, 1].plot(trace.time, utilization)
chosen = cases["low grip, gated + traction + delay"]
for value, name in (
    (chosen.requested, "Requested"),
    (chosen.limited, "Limited"),
    (chosen.applied, "Applied"),
):
    axes[1, 0].plot(chosen.time, value, label=name)
axes[0, 0].legend(fontsize=8)
axes[1, 0].legend()
for ax, label in zip(
    axes.flat, ("Lean (deg)", "Slip (m/s)", "Total torque (N·m)", "|Fₜ| / N")
):
    ax.set(xlabel="Time (s)", ylabel=label)
plt.show()

In [ ]:
# Recovery map for balance-only under a conservative friction belief.
angle_grid = [3.0, 8.0, 15.0]
friction_grid = [0.15, 0.30, 0.65]
recovered = np.zeros((len(angle_grid), len(friction_grid)), dtype=bool)
for i, angle in enumerate(angle_grid):
    for j, friction in enumerate(friction_grid):
        trace = run(
            friction,
            "gated",
            0.8 * friction,
            angle=angle,
            delay=1,
            noise=True,
            hold=False,
        )
        recovered[i, j] = (
            trace.outcome == "completed"
            and abs(np.rad2deg(trace.state[-1, 2])) <= s["recovered_angle_deg"]
            and abs(trace.state[-1, 1]) <= s["recovered_speed_m_s"]
        )
print(
    "Rows are initial lean (deg):", angle_grid, "columns are friction:", friction_grid
)
print("Recovery within declared horizon and tolerances:\n", recovered)

In [ ]:
from dataclasses import replace

sensitivity = s["sensitivity"]
mu = sensitivity["actual_friction"]
perturbed = replace(
    example,
    upper_inertia_kg_m2=example.upper_inertia_kg_m2
    * sensitivity["upper_inertia_scale"],
)
for ratio in sensitivity["friction_belief_ratios"]:
    baseline = run(
        mu,
        "gated",
        ratio * mu,
        angle=sensitivity["test_lean_deg"],
        noise=True,
        hold=False,
    )
    stressed = run(
        mu,
        "gated",
        ratio * mu,
        angle=sensitivity["test_lean_deg"],
        noise=True,
        hold=False,
        delay=sensitivity["delayed_samples"],
        plant=perturbed,
        motor_bound=sensitivity["reduced_motor_bound_N_m"],
    )
    assert (
        np.max(abs(stressed.applied)) <= sensitivity["reduced_motor_bound_N_m"] + 1e-12
    )
    print(
        "Friction belief/true ratio:",
        ratio,
        "baseline/stressed outcome:",
        baseline.outcome,
        stressed.outcome,
        "peak slip (m/s):",
        round(np.max(abs(baseline.slip)), 3),
        round(np.max(abs(stressed.slip)), 3),
    )